# 04: Açıklanabilir Yapay Zeka (XAI) (Kişi 4)

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini

**Bu notebook'ta yapılanlar:**
1. Açıklanabilir yapay zeka yöntemlerinin araştırılması
2. Projeye uygun yöntemlerin seçilmesi (SHAP ve LIME) ve gerekçesi
3. Ekibin ortak kullanacağı XAI fonksiyonunun yazılması (`xai_fonksiyonu.py`)
4. Fonksiyonun ekip tarafından nasıl kullanılacağı ve sonuçların nasıl yorumlanacağı

## 1. Kurulum

In [ ]:
# Colab'da shap ve lime hazır gelmez, bir kez kurulur
%pip install -q shap lime

## 2. Açıklanabilir Yapay Zeka (XAI) nedir, neden gerekli?

Random Forest, XGBoost veya SVM gibi modeller yüksek doğruluk verse de **nasıl karar verdiklerini doğrudan göstermezler** (kara kutu modeller). Sağlık gibi alanlarda bir modelin "bu hastada inme riski yüksek" demesi yetmez; doktorun **neden** bu sonuca varıldığını görebilmesi gerekir. XAI yöntemleri, modelin kararlarını insanların anlayabileceği şekilde açıklar.

XAI yöntemleri iki ana gruba ayrılır:

| | Global açıklama | Lokal açıklama |
|---|---|---|
| **Soru** | Model genel olarak hangi özelliklere dayanıyor? | Model **bu hasta** için neden bu kararı verdi? |
| **Örnek** | "Yaş ve büyük tansiyon en önemli iki özellik" | "Bu hastada risk yüksek çünkü yaşı 65 ve tansiyonu 180" |

### İncelenen yöntemler

| Yöntem | Türü | Kısa açıklama |
|---|---|---|
| **Feature Importance** (ağaç modelleri) | Global | Ağaçlardaki bölünmelere göre önem. Hızlı ama yönü (riski artırıyor mu azaltıyor mu) göstermez, yanıltıcı olabilir |
| **Permutation Importance** | Global | Bir özelliği karıştırınca performans ne kadar düşüyor? Modelden bağımsız ama yön bilgisi yok |
| **SHAP** (SHapley Additive exPlanations) | Global + lokal | Oyun teorisindeki Shapley değerlerine dayanır. Her özelliğin tahmine katkısını adil şekilde dağıtır, yönü de gösterir |
| **LIME** (Local Interpretable Model-agnostic Explanations) | Lokal | Tek bir örneğin etrafında yapay veriler üretip basit bir doğrusal model kurar ve kararı onunla açıklar |

### Seçilen yöntemler: SHAP + LIME

- **SHAP** global açıklama için seçildi: matematiksel temeli güçlüdür, hem önem sırasını hem de etkinin yönünü gösterir. Ağaç modelleri için özel hızlı bir sürümü (TreeExplainer) vardır.
- **LIME** lokal açıklama için seçildi: modelden bağımsızdır (her modelde aynı şekilde çalışır) ve tek bir hasta için anlaşılır bir açıklama verir.
- İki yöntemi birlikte kullanmak, sonuçların birbirini doğrulayıp doğrulamadığını görmemizi sağlar.

### SHAP grafikleri nasıl okunur? (ekip için rehber)

- **Bar grafiği:** Çubuk ne kadar uzunsa özellik o kadar önemli.
- **Özet (beeswarm) grafiği:** Her nokta bir hasta. **Kırmızı** = özelliğin değeri yüksek, **mavi** = düşük. Nokta **sağdaysa** inme riskini **artırıyor**, soldaysa azaltıyor. Örnek: AGE satırında kırmızı noktalar sağdaysa "yaş arttıkça risk artıyor" denir.
- **LIME grafiği:** Yeşil çubuklar "inme var" kararını destekleyen, kırmızı çubuklar karşı çıkan koşullardır (örn. `AGE > 60`).

## 3. Ortak XAI fonksiyonu

Aşağıdaki hücre fonksiyonları `xai_fonksiyonu.py` dosyasına yazar. **Ekipteki herkes** bu dosyayı kendi notebook'una yükleyip tek satırla kullanır:

```python
from xai_fonksiyonu import xai_acikla
sonuc = xai_acikla(model, X_train, X_test, model_adi="SVM")
```

**Ekibe notlar:**
- Model `predict_proba` desteklemeli. **SVM için `SVC(probability=True)` kullanılmalı.**
- Model bir pipeline olabilir (ör. StandardScaler + SMOTE + model). SMOTE adımı otomatik atlanır.
- Fonksiyon model türünü kendisi algılar: ağaç modellerinde TreeExplainer, lineer modellerde LinearExplainer, KNN/SVM'de KernelExplainer kullanır. KernelExplainer yavaş olduğu için test verisinden 150 satırlık örneklem alınır.
- Lineer ve XGBoost modellerinde SHAP değerleri log-odds, diğerlerinde olasılık ölçeğindedir. Modelleri karşılaştırırken ham sayılar yerine `modelleri_karsilastir()` fonksiyonunun verdiği yüzdeler kullanılmalıdır.

In [ ]:
%%writefile xai_fonksiyonu.py
"""
xai_fonksiyonu.py
Ortak Açıklanabilir Yapay Zeka (XAI) fonksiyonları - İnme Tespiti Projesi

Herkes kendi modelini eğittikten sonra bu dosyadaki fonksiyonları çağırır:

    from xai_fonksiyonu import xai_acikla
    sonuc = xai_acikla(model, X_train, X_test, model_adi="Random Forest")

Kullanılan yöntemler:
  - SHAP (global): Model genel olarak hangi özelliklere önem veriyor?
  - LIME (lokal):  Tek bir hasta için model neden "inme var/yok" dedi?

Gereksinimler:
  - X_train ve X_test sayısal pandas DataFrame olmalı (encoding yapılmış).
  - Model predict_proba desteklemeli. SVM için: SVC(probability=True) !
  - Model düz bir sklearn modeli veya imblearn/sklearn Pipeline olabilir
    (ör. StandardScaler + SMOTE + model). SMOTE adımı otomatik atlanır.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.pipeline import Pipeline as SkPipeline

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
except ImportError:  # imblearn yoksa sadece sklearn pipeline desteklenir
    ImbPipeline = SkPipeline

SINIF_ISIMLERI = ["İnme Yok", "İnme Var"]
AGAC_MODELLERI = ("RandomForest", "ExtraTrees", "DecisionTree",
                  "GradientBoosting", "XGB", "LGBM", "CatBoost")
LINEER_MODELLER = ("LogisticRegression", "LinearSVC", "RidgeClassifier", "SGDClassifier")


# ----------------------------------------------------------------------------
# Yardımcı fonksiyonlar
# ----------------------------------------------------------------------------
def _pipeline_ayir(model, X):
    """Pipeline ise son modeli ve dönüştürülmüş X'i döndürür.
    SMOTE gibi örnekleyiciler transform sırasında zaten atlanır."""
    if isinstance(model, (SkPipeline, ImbPipeline)):
        tahminci = model.steps[-1][1]
        X_donusmus = X
        for _, adim in model.steps[:-1]:
            if adim is None or adim == "passthrough" or hasattr(adim, "fit_resample"):
                continue  # SMOTE gibi örnekleyiciler sadece eğitimde çalışır
            X_donusmus = adim.transform(X_donusmus)
        X_donusmus = pd.DataFrame(np.asarray(X_donusmus), columns=X.columns, index=X.index)
        return tahminci, X_donusmus
    return model, X


def _model_turu(tahminci):
    isim = type(tahminci).__name__
    if any(k in isim for k in AGAC_MODELLERI):
        return "agac"
    if any(k in isim for k in LINEER_MODELLER):
        return "lineer"
    return "diger"  # KNN, SVM vb.


def _pozitif_sinif(shap_degerleri):
    """SHAP çıktısından 'İnme Var' (sınıf 1) değerlerini seçer."""
    deger = shap_degerleri.values if hasattr(shap_degerleri, "values") else shap_degerleri
    if isinstance(deger, list):  # eski SHAP sürümleri
        return deger[1]
    deger = np.asarray(deger)
    if deger.ndim == 3:  # (örnek, özellik, sınıf)
        return deger[:, :, 1]
    return deger


# ----------------------------------------------------------------------------
# 1) SHAP: Global açıklama
# ----------------------------------------------------------------------------
def shap_global(model, X_train, X_test, model_adi="Model", orneklem=150,
                arka_plan=50, grafik=True, random_state=42):
    """Modelin genel olarak hangi özelliklere dayandığını gösterir.

    Model türüne göre doğru SHAP açıklayıcısını otomatik seçer:
      - Ağaç modelleri (RF, XGBoost...) -> TreeExplainer (hızlı, tüm test verisi)
      - Lineer modeller (LR)            -> LinearExplainer (hızlı)
      - Diğerleri (KNN, SVM)            -> KernelExplainer (yavaş, örneklem kullanılır)

    Döndürür: özellik önem tablosu (ortalama |SHAP|), büyükten küçüğe.
    """
    tahminci, Xtr = _pipeline_ayir(model, X_train)
    _, Xte = _pipeline_ayir(model, X_test)
    tur = _model_turu(tahminci)

    if tur == "agac":
        aciklayici = shap.TreeExplainer(tahminci)
        Xte_kullan = Xte
        degerler = _pozitif_sinif(aciklayici.shap_values(Xte_kullan))
    elif tur == "lineer":
        aciklayici = shap.LinearExplainer(tahminci, Xtr)
        Xte_kullan = Xte
        degerler = _pozitif_sinif(aciklayici.shap_values(Xte_kullan))
    else:
        # KernelExplainer yavaş olduğu için test verisinden örneklem alınır
        n = min(orneklem, len(Xte))
        Xte_kullan = Xte.sample(n=n, random_state=random_state)
        arka = shap.kmeans(Xtr, min(arka_plan, len(Xtr)))
        f = lambda x: tahminci.predict_proba(pd.DataFrame(x, columns=Xtr.columns))[:, 1]
        aciklayici = shap.KernelExplainer(f, arka)
        degerler = aciklayici.shap_values(Xte_kullan, silent=True)
        degerler = np.asarray(degerler)

    onem = (pd.DataFrame({"ozellik": Xte_kullan.columns,
                          "ortalama_abs_shap": np.abs(degerler).mean(axis=0)})
            .sort_values("ortalama_abs_shap", ascending=False)
            .reset_index(drop=True))

    if grafik:
        plt.figure()
        shap.summary_plot(degerler, Xte_kullan, plot_type="bar", show=False)
        plt.title(f"{model_adi}: SHAP Özellik Önemi")
        plt.tight_layout()
        plt.show()

        plt.figure()
        shap.summary_plot(degerler, Xte_kullan, show=False)
        plt.title(f"{model_adi}: SHAP Özet Grafiği (İnme Var sınıfı)")
        plt.tight_layout()
        plt.show()

    return onem


# ----------------------------------------------------------------------------
# 2) LIME: Lokal açıklama (tek hasta)
# ----------------------------------------------------------------------------
def lime_lokal(model, X_train, X_test, indeks=None, model_adi="Model",
               ozellik_sayisi=8, grafik=True, random_state=42):
    """Tek bir hasta için modelin kararını açıklar.

    indeks: X_test içindeki satır sırası (0, 1, 2...). Verilmezse modelin
            en yüksek inme olasılığı verdiği hasta seçilir.
    Not: LIME modelin kendisini (pipeline dahil) ham X üzerinde kullanır,
         bu yüzden her model türünde aynı şekilde çalışır.

    Döndürür: (özellik koşulu, etki) listesi.
    """
    if indeks is None:
        olasiliklar = model.predict_proba(X_test)[:, 1]
        indeks = int(np.argmax(olasiliklar))

    aciklayici = LimeTabularExplainer(
        training_data=np.asarray(X_train),
        feature_names=list(X_train.columns),
        class_names=SINIF_ISIMLERI,
        mode="classification",
        discretize_continuous=True,
        random_state=random_state,
    )
    f = lambda x: model.predict_proba(pd.DataFrame(x, columns=X_train.columns))
    hasta = X_test.iloc[indeks].values
    aciklama = aciklayici.explain_instance(hasta, f, num_features=ozellik_sayisi, labels=(1,))

    olasilik = f(hasta.reshape(1, -1))[0, 1]
    print(f"[{model_adi}] Test hastası #{indeks}: İnme olasılığı = {olasilik:.2%}")

    if grafik:
        fig = aciklama.as_pyplot_figure(label=1)
        plt.title(f"{model_adi}: LIME Açıklaması (hasta #{indeks})")
        plt.tight_layout()
        plt.show()

    return aciklama.as_list(label=1)


# ----------------------------------------------------------------------------
# 3) Tek çağrıda hepsi
# ----------------------------------------------------------------------------
def xai_acikla(model, X_train, X_test, model_adi="Model", lime_indeks=None, grafik=True):
    """Herkesin çağıracağı ana fonksiyon: SHAP (global) + LIME (lokal)."""
    print(f"===== {model_adi}: XAI Analizi =====")
    onem = shap_global(model, X_train, X_test, model_adi=model_adi, grafik=grafik)
    print("\nEn önemli 5 özellik (SHAP):")
    print(onem.head(5).to_string(index=False))
    print()
    lime_sonuc = lime_lokal(model, X_train, X_test, indeks=lime_indeks,
                            model_adi=model_adi, grafik=grafik)
    return {"shap_onem": onem, "lime": lime_sonuc}


def modelleri_karsilastir(sonuclar):
    """Aşama 3 için: {model_adi: xai_acikla çıktısı} sözlüğünden,
    her modelin SHAP önem sıralamasını yan yana koyan tablo üretir."""
    tablo = {}
    for ad, s in sonuclar.items():
        o = s["shap_onem"].copy()
        o["ortalama_abs_shap"] = o["ortalama_abs_shap"] / o["ortalama_abs_shap"].sum()
        tablo[ad] = o.set_index("ozellik")["ortalama_abs_shap"]
    df = pd.DataFrame(tablo).fillna(0)
    return df.sort_values(df.columns[0], ascending=False)

In [ ]:
from xai_fonksiyonu import xai_acikla, shap_global, lime_lokal, modelleri_karsilastir

# Colab'da dosyayı ekiple paylaşmak için indir
try:
    from google.colab import files
    files.download("xai_fonksiyonu.py")
except ImportError:
    pass

## 4. Fonksiyonun içindeki adımlar

| Fonksiyon | Ne yapar | Ne döndürür |
|---|---|---|
| `shap_global(model, X_train, X_test)` | Model türünü algılar, uygun SHAP açıklayıcısını seçer, bar ve özet grafiklerini çizer | Özelliklerin ortalama mutlak SHAP değerine göre önem tablosu |
| `lime_lokal(model, X_train, X_test, indeks)` | Seçilen hasta için LIME açıklaması üretir. İndeks verilmezse modelin en riskli gördüğü hastayı seçer | (koşul, etki) listesi, ör. `("AGE > 60", +0.12)` |
| `xai_acikla(model, X_train, X_test, model_adi)` | İkisini birlikte çalıştırır | `{"shap_onem": ..., "lime": ...}` |
| `modelleri_karsilastir(sonuclar)` | Birden fazla modelin SHAP önemlerini yüzdeye çevirip yan yana koyar | Karşılaştırma tablosu |

**Model türüne göre seçilen SHAP açıklayıcısı:**

| Model | Açıklayıcı | Hız |
|---|---|---|
| Random Forest, XGBoost, Decision Tree | `TreeExplainer` | Çok hızlı, tüm test verisi kullanılır |
| Logistic Regression | `LinearExplainer` | Çok hızlı |
| KNN, SVM | `KernelExplainer` | Yavaş, test verisinden 150 satırlık örneklem alınır |

## 5. Ekip için kullanım

Her ekip üyesi kendi model notebook'unun sonunda şu adımları uygular:

1. `xai_fonksiyonu.py` dosyasını Colab'da sol menüdeki klasör simgesinden yükler.
2. Kendi en iyi modeli ile fonksiyonu çağırır:

```python
%pip install -q shap lime
from xai_fonksiyonu import xai_acikla

sonuc = xai_acikla(en_iyi_model, X_train, X_test, model_adi="SVM")
sonuc["shap_onem"].head(10)   # en önemli 10 özellik
```

3. Grafikleri Bölüm 2'deki okuma rehberine göre yorumlar ve kendi raporuna yazar.

**Aşama 3: tüm modellerin karşılaştırılması.** Herkes modelini bitirdiğinde sonuçlar bir sözlükte toplanır:

```python
from xai_fonksiyonu import modelleri_karsilastir

sonuclar = {
    "Logistic Regression": xai_acikla(lr_model, X_train, X_test, "Logistic Regression", grafik=False),
    "KNN": xai_acikla(knn_model, X_train, X_test, "KNN", grafik=False),
    "SVM": xai_acikla(svm_model, X_train, X_test, "SVM", grafik=False),
    "Random Forest": xai_acikla(rf_model, X_train, X_test, "Random Forest", grafik=False),
}
tablo = modelleri_karsilastir(sonuclar)
tablo.plot(kind="barh", figsize=(8, 6))
```

Bu tablo, farklı modellerin aynı özelliklere mi yoksa farklı özelliklere mi dayandığını gösterir. Tüm modellerin ortak olarak öne çıkardığı özellikler, inme riski için en güvenilir göstergeler olarak yorumlanabilir.

## 6. Özet (rapor için)

- Kara kutu modellerin kararlarını açıklamak için XAI yöntemleri incelendi: feature importance, permutation importance, SHAP ve LIME.
- Global açıklama için **SHAP**, lokal (tek hasta) açıklama için **LIME** seçildi. SHAP oyun teorisine dayanan sağlam matematiksel temeli ve etkinin yönünü göstermesi, LIME ise modelden bağımsız olması ve tek hasta düzeyinde anlaşılır açıklama vermesi nedeniyle tercih edildi.
- Tüm ekibin kullanabileceği, model türünü otomatik algılayan ortak bir XAI fonksiyonu (`xai_fonksiyonu.py`) yazıldı.
- Her model, eğitildikten sonra bu fonksiyonla açıklanacak ve modellerin özellik önemleri `modelleri_karsilastir` ile karşılaştırılacaktır.